# 실습 2. 검색 품질 진단 — `top_k` · 임베딩 모델

## 실습 목표

검색이 **"왜 틀리는가"** 를 데이터로 진단합니다(교안 2장). 핵심 원칙은
**"점수를 먼저 찍어라"** — 답을 보기 전에 무엇이 몇 점으로 검색됐는지부터 봅니다.

- (A) `top_k` 를 바꿔 점수·결과 변화 관찰
- (B) **임베딩 모델 교체**(다국어 e5 vs 영어 위주 MiniLM)로 한국어 검색 품질 비교
- 결론: **RAG 첫 튜닝 레버는 임베딩 교체**

## 1. 준비 — 진단용 질문과 인덱싱 함수

`similarity_search_with_relevance_scores` 는 **(문서, 0~1 점수)** 를 함께 돌려줍니다.
점수 분포가 임베딩/청킹/top_k 중 어디가 문제인지 가리켜 줍니다.

In [ ]:
from _common import sample_documents, get_embeddings, EMB_KO, EMB_EN
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

QUESTIONS = [
    "RAG와 파인튜닝의 차이는?",
    "한국어 문서엔 어떤 임베딩을 써야 하나?",
    "환각을 줄이려면 프롬프트를 어떻게 써야 하나?",
    "벡터 데이터베이스에는 어떤 종류가 있나?",
]

def index(model_name):
    chunks = RecursiveCharacterTextSplitter(
        chunk_size=300, chunk_overlap=50).split_documents(sample_documents())
    col = "day21_sq_" + model_name.split("/")[-1].replace(".", "")
    print(col)
    return Chroma.from_documents(chunks, get_embeddings(model_name), collection_name=col)

def show(model_name, k):
    vs = index(model_name)
    print(f"##### 임베딩={model_name.split('/')[-1]}  top_k={k} #####")
    for q in QUESTIONS:
        hits = vs.similarity_search_with_relevance_scores(q, k=k)
        tops = ", ".join(f"{d.metadata['topic']}({s:.2f})" for d, s in hits)
        print(f"  Q: {q}\n     → {tops}")
    vs.delete_collection()

print("준비 완료")

/Users/bkk/프로젝트/yeardream/.venv-1/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


준비 완료


## 2. [A] `top_k` 비교 — 같은 임베딩(e5), k만 변경

정답이 **몇 번째에 몇 점으로** 오는지 봅니다.

In [ ]:
show(EMB_KO, k=1)
print()       #"intfloat/multilingual-e5-small"  ~470MB, 다국어(한국어 양호) — 기본
show(EMB_KO, k=3)

day21_sq_multilingual-e5-small


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6735.26it/s]


##### 임베딩=multilingual-e5-small  top_k=1 #####
  Q: RAG와 파인튜닝의 차이는?
     → 파인튜닝(0.84)
  Q: 한국어 문서엔 어떤 임베딩을 써야 하나?
     → 임베딩(0.86)
  Q: 환각을 줄이려면 프롬프트를 어떻게 써야 하나?
     → 프롬프트(0.83)
  Q: 벡터 데이터베이스에는 어떤 종류가 있나?
     → 벡터DB(0.84)

day21_sq_multilingual-e5-small
##### 임베딩=multilingual-e5-small  top_k=3 #####
  Q: RAG와 파인튜닝의 차이는?
     → 파인튜닝(0.84), RAG(0.80), 평가(0.78)
  Q: 한국어 문서엔 어떤 임베딩을 써야 하나?
     → 임베딩(0.86), 벡터DB(0.77), 청킹(0.76)
  Q: 환각을 줄이려면 프롬프트를 어떻게 써야 하나?
     → 프롬프트(0.83), RAG(0.76), 청킹(0.73)
  Q: 벡터 데이터베이스에는 어떤 종류가 있나?
     → 벡터DB(0.84), 임베딩(0.73), 트랜스포머(0.72)


**포인트**

- 정답이 **1위에 0.8대로 또렷** → k=1로도 맞힘
- k=3은 보조 근거(연관 주제)를 더해 답변 풍부도↑ 하지만 **무관 청크·토큰 비용↑**
- 정답이 상위에 또렷하면 작은 k, 분산돼 있으면 큰 k — **점수 분포를 보고 정한다**

## 3. [B] 임베딩 모델 비교 — e5 vs MiniLM (오늘의 하이라이트)

같은 문서·코드, **임베딩만 교체**해 한국어 질문을 검색합니다.

In [3]:
print("===== 다국어 e5 (한국어 양호) =====")
show(EMB_KO, k=3)                            #"intfloat/multilingual-e5-small"  ~470MB, 다국어(한국어 양호) — 기본
print("\n===== 영어 위주 all-MiniLM (한국어 약함) =====")
show(EMB_EN, k=3)

===== 다국어 e5 (한국어 양호) =====
day21_sq_multilingual-e5-small
##### 임베딩=multilingual-e5-small  top_k=3 #####
  Q: RAG와 파인튜닝의 차이는?
     → 파인튜닝(0.84), RAG(0.80), 평가(0.78)
  Q: 한국어 문서엔 어떤 임베딩을 써야 하나?
     → 임베딩(0.86), 벡터DB(0.77), 청킹(0.76)
  Q: 환각을 줄이려면 프롬프트를 어떻게 써야 하나?
     → 프롬프트(0.83), RAG(0.76), 청킹(0.73)
  Q: 벡터 데이터베이스에는 어떤 종류가 있나?
     → 벡터DB(0.84), 임베딩(0.73), 트랜스포머(0.72)

===== 영어 위주 all-MiniLM (한국어 약함) =====
day21_sq_all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6369.81it/s]


##### 임베딩=all-MiniLM-L6-v2  top_k=3 #####
  Q: RAG와 파인튜닝의 차이는?
     → RAG(0.35), 파인튜닝(0.28), 에이전트(0.25)
  Q: 한국어 문서엔 어떤 임베딩을 써야 하나?
     → 프롬프트(0.17), 트랜스포머(0.12), RAG(0.09)
  Q: 환각을 줄이려면 프롬프트를 어떻게 써야 하나?
     → 프롬프트(0.13), 에이전트(0.13), RAG(0.13)
  Q: 벡터 데이터베이스에는 어떤 종류가 있나?
     → 에이전트(0.17), 평가(0.14), 트랜스포머(0.13)


**포인트 (오늘의 하이라이트)**

- 임베딩만 바꿨더니 e5는 정답이 **0.8대로 또렷이 1위**, MiniLM은 **0.1~0.3대로 뭉개지고 1위가 자주 틀림**
- 점수의 **절대값**도 신호 — 0.1대가 나오면 **임베딩 부적합**을 의심
- "검색이 틀린다"의 가장 흔한 진짜 원인은 **임베딩 선택** → 첫 튜닝 레버는 임베딩 교체

# [TODO]

## [TODO] 1. 진단 질문 추가

`QUESTIONS` 리스트에 **"청크가 너무 크면 무슨 문제가 생기나?"** 질문을 추가하고,
e5 (`EMB_KO`)로 `top_k=3` 검색 결과를 출력하세요.

In [4]:
# [TODO] 1: QUESTIONS 에 질문 추가
# [TODO] 여기에 코드를 작성하세요.
QUESTIONS.append("청크가 너무 크면 무슨 문제가 생기지?")

show(EMB_KO, k=3)

day21_sq_multilingual-e5-small
##### 임베딩=multilingual-e5-small  top_k=3 #####
  Q: RAG와 파인튜닝의 차이는?
     → 파인튜닝(0.84), RAG(0.80), 평가(0.78)
  Q: 한국어 문서엔 어떤 임베딩을 써야 하나?
     → 임베딩(0.86), 벡터DB(0.77), 청킹(0.76)
  Q: 환각을 줄이려면 프롬프트를 어떻게 써야 하나?
     → 프롬프트(0.83), RAG(0.76), 청킹(0.73)
  Q: 벡터 데이터베이스에는 어떤 종류가 있나?
     → 벡터DB(0.84), 임베딩(0.73), 트랜스포머(0.72)
  Q: 청크가 너무 크면 무슨 문제가 생기지?
     → 청킹(0.81), 프롬프트(0.69), 평가(0.68)


## [TODO] 2. "확신 없는 검색" 경고 함수

`warn_low_score(model_name, question, threshold=0.4)` 함수를 작성하세요.
- 해당 임베딩으로 `question` 을 top-1 검색
- **top-1 점수가 `threshold` 미만**이면 `"⚠ 임베딩 부적합 의심"`, 아니면 `"OK"` 를 반환
- (힌트) `similarity_search_with_relevance_scores(q, k=1)` 의 점수를 사용

In [13]:
def warn_low_score(model_name, question, threshold=0.4):
    # [TODO] 2: top-1 점수가 threshold 미만이면 경고 문자열 반환
    # [TODO] 여기에 코드를 작성하세요.
    vs = index(model_name)
    score = vs.similarity_search_with_relevance_scores(question,k=1)

    print("="*20)
    print(score)
    print("="*20)


    if score[0][1] < threshold:
        return "⚠ 임베딩 부적합 의심"
    return "OK"



q = "한국어 문서엔 어떤 임베딩을 써야 하나?"
print("e5    :", warn_low_score(EMB_KO, q))
print("MiniLM:", warn_low_score(EMB_EN, q))

day21_sq_multilingual-e5-small
[(Document(id='12d19053-affa-483d-b53f-6fb3e300db32', metadata={'topic': '임베딩', 'id': 'doc-embedding'}, page_content='임베딩은 텍스트를 의미가 보존된 고정 길이 벡터로 변환한 것이다. 비슷한 의미의 문장은 벡터 공간에서 가깝게 위치한다. 임베딩 모델의 언어·도메인 적합성은 검색 품질을 좌우하며, 한국어 문서에는 다국어 또는 한국어 특화 임베딩 모델을 써야 검색 정확도가 올라간다.'), 0.8638354073358694)]
e5    : OK
day21_sq_all-MiniLM-L6-v2
[(Document(id='35c4ac84-989a-41ee-9818-5e2973431fa5', metadata={'topic': '프롬프트', 'id': 'doc-prompt'}, page_content="프롬프트 엔지니어링은 모델에게 작업을 지시하는 입력을 설계하는 기술이다. 역할 부여, 예시 제공(few-shot), 단계적 사고 유도 같은 기법으로 응답 품질을 높인다. RAG에서는 검색된 문맥을 프롬프트에 넣고 '문맥에 근거해서만 답하라'고 지시하는 것이 환각을 줄이는 핵심이다."), 0.17442854289297205)]
MiniLM: ⚠ 임베딩 부적합 의심


## 실습 정리

- **점수를 먼저 찍는** 진단 습관을 익힘
- `top_k` 의 recall/precision 트레이드오프 관찰
- **임베딩 교체가 한국어 검색 품질을 좌우** — RAG 튜닝의 첫 레버